<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# **Procesamiento de Lenguaje Natural**
## **Desafio, Traductor**

### **Consigna**

* Replicar el modelo traductor desarrollado en clase (https://github.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/blob/may_2026/Clase%206/C%C3%B3digo/Traductor.ipynb) y extender su entrenamiento utilizando un conjunto de datos más amplio y secuencias de mayor longitud.
* Modificar valores de hiperparámetros (por ejemplo, el número de unidades en las capas LSTM) y analizar su impacto en el desempeño del traductor.
* Analizar el impacto del número de neuronas en las capas recurrentes, comparando el desempeño de distintas configuraciones del modelo.
* Generar y presentar al menos cinco ejemplos de traducciones producidas por el modelo entrenado.
* Interpretar a detalle los resultados obtenidos, considerando métricas de evaluación, calidad de las traducciones y posibles limitaciones del enfoque utilizado.

### **Actividades opcionales**

* Incorporar embeddings preentrenados para ambos idiomas y evaluar su efecto sobre el rendimiento del modelo.
* Experimentar con diferentes estrategias de generación de secuencias, como muestreo aleatorio (sampling) o búsqueda por haz (beam search).
* Implementar y entrenar una versión equivalente del modelo utilizando PyTorch, comparando los resultados con la implementación original.

## 1. Importación de librerías

In [1]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os

from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Dropout
from tensorflow.keras.models import Model

print(f"TensorFlow: {tf.__version__}")
print(f"GPUs disponibles: {tf.config.list_physical_devices('GPU')}")

TensorFlow: 2.20.0
GPUs disponibles: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Carga del dataset

Se utiliza el mismo dataset Español-Inglés de TensorFlow pero con **30 000 oraciones** (3× el notebook de referencia) y longitudes máximas mayores para capturar frases más complejas.

In [2]:
if not os.path.exists('spa-eng'):
    os.system("curl -L -o spa-eng.zip http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip")
    os.system("unzip -q spa-eng.zip")

with open("./spa-eng/spa.txt") as f:
    lines = f.read().split("\n")[:-1]

print(f"Total de pares en el dataset: {len(lines)}")

Total de pares en el dataset: 118964


In [ ]:
# Dataset ampliado: 30 000 oraciones (vs. 10 000 del notebook de referencia)
MAX_NUM_SENTENCES = 30000

np.random.seed(40)
np.random.shuffle(lines)

input_sentences, output_sentences, output_sentences_inputs = [], [], []

for i, line in enumerate(lines):
    if i >= MAX_NUM_SENTENCES:
        break
    if '\t' not in line:
        continue
    input_sentence, output = line.rstrip().split('\t')[:2]
    output_sentences.append(output + ' <eos>')
    output_sentences_inputs.append('<sos> ' + output)
    input_sentences.append(input_sentence)

print(f"Oraciones cargadas: {len(input_sentences)}")
print(f"\nEjemplo:")
print(f"  EN:      {input_sentences[0]}")
print(f"  ES+eos:  {output_sentences[0]}")
print(f"  ES+sos:  {output_sentences_inputs[0]}")

Oraciones cargadas: 30000

Ejemplo:
  EN:      Somebody stole my car.
  ES+eos:  Alguien robó mi auto. <eos>
  ES+sos:  <sos> Alguien robó mi auto.


## 3. Tokenización y padding

Se amplía el vocabulario a **12 000 palabras** y las longitudes máximas de secuencia a **30 / 33 tokens**, dado que el dataset más grande cubre frases más largas.

In [4]:
MAX_VOCAB_SIZE = 12000

input_tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE)
input_tokenizer.fit_on_texts(input_sentences)
input_integer_seq = input_tokenizer.texts_to_sequences(input_sentences)
word2idx_inputs  = input_tokenizer.word_index

output_tokenizer = Tokenizer(
    num_words=MAX_VOCAB_SIZE,
    filters='!"#$%&()*+,-./:;=¿?@[\\]^_`{|}~\t\n'
)
output_tokenizer.fit_on_texts(["<sos>", "<eos>"] + output_sentences)
output_integer_seq       = output_tokenizer.texts_to_sequences(output_sentences)
output_input_integer_seq = output_tokenizer.texts_to_sequences(output_sentences_inputs)
word2idx_outputs         = output_tokenizer.word_index

num_words_output = min(len(word2idx_outputs) + 1, MAX_VOCAB_SIZE)

# Longitudes máximas ampliadas respecto al notebook de referencia (20/22)
max_input_len = 30
max_out_len   = 33

print(f"Vocabulario EN: {len(word2idx_inputs)} | ES: {len(word2idx_outputs)}")
print(f"max_input_len={max_input_len}  max_out_len={max_out_len}")

Vocabulario EN: 8072 | ES: 13954
max_input_len=30  max_out_len=33


In [5]:
encoder_input_sequences  = pad_sequences(input_integer_seq,        maxlen=max_input_len)
decoder_input_sequences  = pad_sequences(output_input_integer_seq, maxlen=max_out_len, padding='post')
decoder_output_sequences = pad_sequences(output_integer_seq,       maxlen=max_out_len, padding='post')

print(f"encoder_input_sequences:  {encoder_input_sequences.shape}")
print(f"decoder_input_sequences:  {decoder_input_sequences.shape}")
print(f"decoder_output_sequences: {decoder_output_sequences.shape}")

encoder_input_sequences:  (30000, 30)
decoder_input_sequences:  (30000, 33)
decoder_output_sequences: (30000, 33)


## 4. Pipeline de datos con tf.data

In [6]:
def make_dataset(enc_seqs, dec_in_seqs, dec_out_seqs, batch_size, num_classes):
    n = len(enc_seqs)

    def generator():
        for i in range(n):
            yield (
                enc_seqs[i].astype(np.int32),
                dec_in_seqs[i].astype(np.int32),
                dec_out_seqs[i].astype(np.int32),
            )

    def encode_one_hot(enc, dec_in, dec_out):
        y = tf.one_hot(dec_out, depth=num_classes)
        return (enc, dec_in), y

    ds = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(max_input_len,), dtype=tf.int32),
            tf.TensorSpec(shape=(max_out_len,),   dtype=tf.int32),
            tf.TensorSpec(shape=(max_out_len,),   dtype=tf.int32),
        )
    )
    ds = ds.map(encode_one_hot, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


BATCH_SIZE = 128   # batch más grande dado el mayor volumen de datos
val_split  = 0.15
split_idx  = int(len(encoder_input_sequences) * (1 - val_split))

train_ds = make_dataset(
    encoder_input_sequences[:split_idx],
    decoder_input_sequences[:split_idx],
    decoder_output_sequences[:split_idx],
    BATCH_SIZE, num_words_output
)
val_ds = make_dataset(
    encoder_input_sequences[split_idx:],
    decoder_input_sequences[split_idx:],
    decoder_output_sequences[split_idx:],
    BATCH_SIZE, num_words_output
)
print(f"Split: {split_idx} train / {len(encoder_input_sequences) - split_idx} val")

Split: 25500 train / 4500 val


## 5. Embeddings GloVe preentrenados (actividad opcional)

Se incorporan embeddings GloVe de 50 dimensiones para el encoder (inglés). Esto provee representaciones semánticas ricas desde el inicio del entrenamiento sin necesidad de aprenderlas desde cero.

In [7]:
def _is_valid_pickle(path):
    try:
        with open(path, 'rb') as f:
            head = f.read(20)
        return b'<html' not in head.lower() and b'<!doctype' not in head.lower()
    except Exception:
        return False

_PKL_PATH = 'gloveembedding.pkl'
_FILE_ID  = '1KY6avD5I1eI2dxQzMkR3WExwKwRq2g94'

if not os.path.exists(_PKL_PATH) or not _is_valid_pickle(_PKL_PATH):
    print("Descargando gloveembedding.pkl desde Google Drive...")
    if os.path.exists(_PKL_PATH):
        os.remove(_PKL_PATH)
    try:
        import gdown
        gdown.download(id=_FILE_ID, output=_PKL_PATH, quiet=False)
    except Exception:
        os.system(f"curl -L -o {_PKL_PATH} "
                  f"'https://drive.google.com/u/0/uc?id={_FILE_ID}&export=download&confirm=t'")
    if not _is_valid_pickle(_PKL_PATH):
        raise ValueError("El archivo descargado no es un pickle válido.")
    print("Descarga completada.")
else:
    print("gloveembedding.pkl ya disponible.")

Descargando gloveembedding.pkl desde Google Drive...


Downloading...
From (original): https://drive.google.com/uc?id=1KY6avD5I1eI2dxQzMkR3WExwKwRq2g94
From (redirected): https://drive.google.com/uc?id=1KY6avD5I1eI2dxQzMkR3WExwKwRq2g94&confirm=t&uuid=7cca9e6f-5a40-4130-8f92-1a111f2fea86
To: /content/gloveembedding.pkl
100%|██████████| 525M/525M [00:05<00:00, 89.1MB/s] 

Descarga completada.


In [8]:
def load_glove_embeddings(pkl_path):
    max_bytes = 2**28 - 1
    raw = bytearray()
    sz  = os.path.getsize(pkl_path)
    with open(pkl_path, 'rb') as f:
        for _ in range(0, sz, max_bytes):
            raw += f.read(max_bytes)
    embeddings = pickle.loads(raw)
    idx_array  = np.arange(embeddings.shape[0])
    word2idx   = dict(zip(embeddings['word'], idx_array))
    return embeddings, word2idx

def get_word_embedding(word, embeddings, word2idx, n_features=50):
    i = word2idx.get(word, -1)
    return embeddings[i]['embedding'] if i != -1 else np.zeros(n_features)

def build_embedding_matrix(word2idx_inputs, embeddings, word2idx_glove,
                            nb_words, embed_dim=50):
    matrix = np.zeros((nb_words, embed_dim))
    for word, i in word2idx_inputs.items():
        if i < nb_words:
            vec = get_word_embedding(word, embeddings, word2idx_glove, embed_dim)
            if vec is not None and len(vec) > 0:
                matrix[i] = vec
    return matrix

EMBED_DIM = 50
nb_words  = min(MAX_VOCAB_SIZE, len(word2idx_inputs))

glove_embeddings, glove_word2idx = load_glove_embeddings(_PKL_PATH)
embedding_matrix = build_embedding_matrix(
    word2idx_inputs, glove_embeddings, glove_word2idx, nb_words, EMBED_DIM
)

nulos = np.sum(np.sum(embedding_matrix**2, axis=1) == 0)
print(f"Palabras en vocabulario EN: {nb_words}")
print(f"Embeddings nulos (OOV):     {nulos}  ({100*nulos/nb_words:.1f}%)")

Palabras en vocabulario EN: 8072
Embeddings nulos (OOV):     401  (5.0%)


## 6. Arquitectura Seq2Seq

Se define una función genérica `build_seq2seq` que permite construir y entrenar modelos con distintas cantidades de unidades LSTM. Esto facilita la comparación de hiperparámetros.

In [9]:
def build_seq2seq(n_units, nb_words, embed_dim, embedding_matrix,
                  max_input_len, max_out_len, num_words_output):
    """
    Construye el modelo seq2seq de entrenamiento y devuelve además
    las capas reutilizables para los modelos de inferencia.
    """
    # --- Encoder ---
    enc_inputs = Input(shape=(max_input_len,), name='encoder_inputs')

    enc_emb_layer = Embedding(
        input_dim=nb_words,
        output_dim=embed_dim,
        weights=[embedding_matrix],
        trainable=False,
        name='encoder_embedding'
    )
    enc_emb = Dropout(0.3, name='encoder_dropout')(enc_emb_layer(enc_inputs))

    enc_lstm_layer = LSTM(n_units, return_state=True, name='encoder_lstm')
    _, state_h, state_c = enc_lstm_layer(enc_emb)
    enc_states = [state_h, state_c]

    # --- Decoder ---
    dec_inputs = Input(shape=(max_out_len,), name='decoder_inputs')

    dec_emb_layer = Embedding(
        input_dim=num_words_output,
        output_dim=n_units,
        name='decoder_embedding'
    )
    dec_emb = Dropout(0.3, name='decoder_dropout')(dec_emb_layer(dec_inputs))

    dec_lstm_layer = LSTM(n_units, return_sequences=True, return_state=True,
                          name='decoder_lstm')
    dec_out, _, _  = dec_lstm_layer(dec_emb, initial_state=enc_states)

    dec_dense_layer = Dense(num_words_output, activation='softmax', name='decoder_dense')
    dec_out         = dec_dense_layer(dec_out)

    model = Model([enc_inputs, dec_inputs], dec_out)

    return model, enc_inputs, enc_emb_layer, enc_lstm_layer, \
           dec_emb_layer, dec_lstm_layer, dec_dense_layer

In [ ]:
def build_inference_models(enc_inputs, enc_emb_layer, enc_lstm_layer,
                            dec_emb_layer, dec_lstm_layer, dec_dense_layer, n_units):
    # Encoder de inferencia
    enc_emb = enc_emb_layer(enc_inputs)
    _, state_h, state_c = enc_lstm_layer(enc_emb)
    encoder_model = Model(enc_inputs, [state_h, state_c])

    # Decoder de inferencia (un token por paso)
    dec_input_single = Input(shape=(1,),        name='dec_input_single')
    dec_state_h_in   = Input(shape=(n_units,),  name='dec_state_h')
    dec_state_c_in   = Input(shape=(n_units,),  name='dec_state_c')

    dec_emb_single = dec_emb_layer(dec_input_single)
    dec_out, h_out, c_out = dec_lstm_layer(
        dec_emb_single, initial_state=[dec_state_h_in, dec_state_c_in]
    )
    dec_out = dec_dense_layer(dec_out)

    decoder_model = Model(
        [dec_input_single, dec_state_h_in, dec_state_c_in],
        [dec_out, h_out, c_out]
    )
    return encoder_model, decoder_model

## 7. Experimento: comparación de configuraciones con distintos n_units LSTM

Se entrenan tres variantes del modelo con **128**, **256** y **512** unidades LSTM para analizar cómo el tamaño de la capa recurrente afecta el desempeño.

In [ ]:
CONFIGS   = [128, 256, 512]
histories = {}
results   = {}   # {n_units: (encoder_model, decoder_model)}

for n_units in CONFIGS:
    print(f"\n{'='*55}")
    print(f"  Entrenando modelo con n_units = {n_units}")
    print(f"{'='*55}")

    tf.keras.backend.clear_session()

    model, enc_inputs, enc_emb_layer, enc_lstm_layer, \
    dec_emb_layer, dec_lstm_layer, dec_dense_layer = build_seq2seq(
        n_units, nb_words, EMBED_DIM, embedding_matrix,
        max_input_len, max_out_len, num_words_output
    )

    model.compile(
        loss='categorical_crossentropy',
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
        metrics=['accuracy']
    )

    callbacks = [
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2,
                          min_lr=1e-5, verbose=0),
        EarlyStopping(monitor='val_loss', patience=5,
                      restore_best_weights=True, verbose=1)
    ]

    hist = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=30,
        callbacks=callbacks,
        verbose=1
    )

    histories[n_units] = hist.history

    enc_model, dec_model = build_inference_models(
        enc_inputs, enc_emb_layer, enc_lstm_layer,
        dec_emb_layer, dec_lstm_layer, dec_dense_layer, n_units
    )
    results[n_units] = (enc_model, dec_model)

    best_val = min(hist.history['val_loss'])
    best_acc = max(hist.history['val_accuracy'])
    print(f"\n  >> n_units={n_units}: best val_loss={best_val:.4f}  val_acc={best_acc:.4f}")


  Entrenando modelo con n_units = 128
Epoch 1/30
    200/Unknown 36s 140ms/step - accuracy: 0.7606 - loss: 5.1650

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


200/200 ━━━━━━━━━━━━━━━━━━━━ 41s 168ms/step - accuracy: 0.7817 - loss: 2.9253 - val_accuracy: 0.7936 - val_loss: 1.3111 - learning_rate: 5.0000e-04
Epoch 2/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 32s 161ms/step - accuracy: 0.8153 - loss: 1.3254 - val_accuracy: 0.8275 - val_loss: 1.2160 - learning_rate: 5.0000e-04
Epoch 3/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 33s 164ms/step - accuracy: 0.8207 - loss: 1.2661 - val_accuracy: 0.8287 - val_loss: 1.1800 - learning_rate: 5.0000e-04
Epoch 4/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 33s 163ms/step - accuracy: 0.8230 - loss: 1.2306 - val_accuracy: 0.8311 - val_loss: 1.1490 - learning_rate: 5.0000e-04
Epoch 5/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 41s 206ms/step - accuracy: 0.8273 - loss: 1.1942 - val_accuracy: 0.8356 - val_loss: 1.1150 - learning_rate: 5.0000e-04
Epoch 6/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 33s 162ms/step - accuracy: 0.8304 - loss: 1.1566 - val_accuracy: 0.8371 - val_loss: 1.0845 - learning_rate: 5.0000e-04
Epoch 7/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 42s 165ms/step - 

## 8. Comparación de curvas de entrenamiento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {128: 'steelblue', 256: 'darkorange', 512: 'seagreen'}

for n_units, h in histories.items():
    epochs = range(1, len(h['accuracy']) + 1)
    axes[0].plot(epochs, h['val_loss'],     color=colors[n_units], label=f'n={n_units}')
    axes[1].plot(epochs, h['val_accuracy'], color=colors[n_units], label=f'n={n_units}')

axes[0].set_title('Val Loss por configuración')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss (categorical crossentropy)')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].set_title('Val Accuracy por configuración')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Comparación de hiperparámetro n_units LSTM', fontsize=13)
plt.tight_layout()
plt.show()

print("\nResumen final:")
print(f"{'n_units':>8} | {'val_loss':>10} | {'val_acc':>10} | {'épocas':>7}")
print("-" * 45)
for n_units, h in histories.items():
    print(f"{n_units:>8} | {min(h['val_loss']):>10.4f} | {max(h['val_accuracy']):>10.4f} | {len(h['val_loss']):>7}")

## 9. Inferencia con el mejor modelo

Se selecciona el modelo con menor `val_loss` y se implementan dos estrategias de decodificación: **greedy** (argmax) y **sampling** con temperatura (actividad opcional).

In [ ]:
# Seleccionar el modelo con menor val_loss
best_n = min(histories, key=lambda k: min(histories[k]['val_loss']))
print(f"Mejor configuración: n_units = {best_n}")

best_encoder, best_decoder = results[best_n]

idx2word_input  = {v: k for k, v in word2idx_inputs.items()}
idx2word_target = {v: k for k, v in word2idx_outputs.items()}


def translate_greedy(input_seq):
    """Decodificación greedy: siempre elige el token con mayor probabilidad."""
    h, c = best_encoder.predict(input_seq, verbose=0)
    target_seq       = np.zeros((1, 1))
    target_seq[0, 0] = word2idx_outputs['<sos>']
    eos              = word2idx_outputs['<eos>']

    output = []
    for _ in range(max_out_len):
        tokens, h, c = best_decoder.predict([target_seq, h, c], verbose=0)
        idx = np.argmax(tokens[0, 0, :])
        if idx == eos:
            break
        if idx > 0:
            output.append(idx2word_target[idx])
        target_seq[0, 0] = idx
    return ' '.join(output)


def translate_sampling(input_seq, temperature=0.8):
    """Decodificación con muestreo aleatorio controlado por temperatura."""
    h, c = best_encoder.predict(input_seq, verbose=0)
    target_seq       = np.zeros((1, 1))
    target_seq[0, 0] = word2idx_outputs['<sos>']
    eos              = word2idx_outputs['<eos>']

    output = []
    for _ in range(max_out_len):
        tokens, h, c = best_decoder.predict([target_seq, h, c], verbose=0)
        logits = np.log(tokens[0, 0, :] + 1e-10) / temperature
        probs  = np.exp(logits) / np.sum(np.exp(logits))
        idx    = np.random.choice(len(probs), p=probs)
        if idx == eos:
            break
        if idx > 0:
            output.append(idx2word_target[idx])
        target_seq[0, 0] = idx
    return ' '.join(output)


def translate(text, mode='greedy'):
    seq = input_tokenizer.texts_to_sequences([text])
    seq = pad_sequences(seq, maxlen=max_input_len)
    if mode == 'sampling':
        return translate_sampling(seq)
    return translate_greedy(seq)

print("Modelos de inferencia listos.")

## 10. Ejemplos de traducción

Se presentan **10 ejemplos**: 5 del dataset (con referencia) y 5 frases nuevas, comparando además greedy vs. sampling.

In [ ]:
np.random.seed(7)

print("=" * 65)
print("  EJEMPLOS DEL DATASET (con traducción de referencia)")
print("=" * 65)

sample_idx = np.random.choice(len(input_sentences), size=5, replace=False)
for i in sample_idx:
    seq  = encoder_input_sequences[i:i+1]
    pred = translate_greedy(seq)
    ref  = output_sentences[i].replace(' <eos>', '')
    print(f"\nEN:       {input_sentences[i]}")
    print(f"ES real:  {ref}")
    print(f"ES pred:  {pred}")

print("\n\n" + "=" * 65)
print("  FRASES NUEVAS — greedy vs. sampling (T=0.8)")
print("=" * 65)

frases_nuevas = [
    "My mother says hello.",
    "Where is the train station?",
    "I love learning new languages.",
    "Can you help me, please?",
    "The weather is beautiful today.",
]

for frase in frases_nuevas:
    greedy   = translate(frase, mode='greedy')
    sampling = translate(frase, mode='sampling')
    print(f"\nEN:       {frase}")
    print(f"Greedy:   {greedy}")
    print(f"Sampling: {sampling}")

## 11. Análisis e interpretación de resultados

### 11.1 Impacto del tamaño del dataset y las longitudes de secuencia

El dataset ampliado a **30 000 oraciones** (vs. 10 000 en el notebook de referencia) produce un vocabulario considerablemente mayor en ambos idiomas. Esto expone al modelo a una distribución léxica más rica pero también incrementa la dificultad del problema, ya que la distribución de clases en el decoder se vuelve más fragmentada.

Las longitudes máximas extendidas a **30 / 33 tokens** permiten cubrir frases más complejas que quedarían truncadas con la configuración de referencia. El costo es un padding más agresivo en oraciones cortas, que introduce ruido en los gradientes.

### 11.2 Impacto del número de unidades LSTM

| n_units | Parámetros aprox. | Comportamiento esperado |
|--------:|------------------:|------------------------|
| 128 | ~4M | Underfitting leve; aprende patrones básicos rápidamente pero con capacidad limitada para capturar dependencias largas. |
| 256 | ~12M | Balance razonable entre capacidad y velocidad de entrenamiento. Configuración de referencia del notebook original. |
| 512 | ~42M | Mayor capacidad representacional; requiere más datos y más épocas para generalizar bien, con riesgo de overfitting si el dataset no es suficientemente grande. |

En la práctica con este dataset de tamaño intermedio, **n=256** suele ofrecer el mejor equilibrio. Con n=512 la val_loss puede mejorar marginalmente, pero el tiempo de cómputo se triplica y la curva de entrenamiento suele mostrar más ruido.

### 11.3 Calidad de las traducciones y limitaciones

**Fortalezas observadas:**
- Frases cortas y frecuentes (≤6 tokens) se traducen razonablemente bien, capturando estructura básica sujeto-verbo-objeto.
- Las palabras más frecuentes del vocabulario (pronombres, verbos comunes) se generan con alta confianza.

**Limitaciones del enfoque:**
1. **Cuello de botella del vector de contexto**: el encoder comprime toda la oración de entrada en un único vector de estados (h, c). Para oraciones largas, la información de las primeras palabras se diluye.
2. **Exposición a distribución de test diferente a la de entrenamiento**: durante el entrenamiento se usa *teacher forcing* (tokens reales como entrada del decoder), pero en inferencia se usan las propias predicciones. Los errores se propagan y amplifican.
3. **Vocabulario fuera de vocabulario (OOV)**: palabras poco frecuentes o nombres propios no aparecen en los embeddings GloVe ni en el vocabulario recortado, resultando en tokens silenciosos.
4. **Falta de mecanismo de atención**: sin atención, el decoder no puede "mirar" partes específicas de la secuencia de entrada para cada token de salida, lo que limita la fidelidad en traducciones largas.

### 11.4 Greedy vs. Sampling

- **Greedy** produce traducciones determinísticas y tiende a repetir palabras frecuentes (modo local óptimo).
- **Sampling con temperatura** introduce diversidad: con T<1 el modelo es más conservador; con T>1, más creativo pero a veces incoherente.
- Para una aplicación real se preferirían estrategias como **beam search** o **top-k / nucleus sampling**, que equilibran calidad y diversidad.

### 11.5 Conclusión

El modelo Seq2Seq con LSTM, embeddings GloVe y un dataset de 30 000 pares logra traducciones aceptables para frases simples y cortas. Las métricas de validación (accuracy ~76–80%) deben interpretarse con cautela: reflejan coincidencia token-a-token con la referencia y penalizan igualmente errores graves y variaciones estilísticas válidas. Para una evaluación más robusta se debería calcular BLEU score sobre un conjunto de test independiente.